In [0]:
import requests

# Configure this secret scope and key before running (see docs/setup.md).
source_sas = dbutils.secrets.get(scope="retail-sales", key="source-sas").strip().lstrip("?")
url = f"https://strsalesdata.blob.core.windows.net/csalesdata/raw/sales.csv?{source_sas}"
response = requests.get(url, timeout=120)
if response.status_code != 200:
    raise RuntimeError(f"Source download failed: HTTP {response.status_code}")
with open("/tmp/sales.csv", "wb") as f:
    f.write(response.content)

In [0]:
# Small CSV demonstration: pandas reads driver-local data; Spark handles transformations.
import pandas as pd

df = spark.createDataFrame(pd.read_csv("/tmp/sales.csv"))
display(df)

Convert Data Types

In [0]:
from pyspark.sql.functions import col

df  = df.withColumn("quantity", col("quantity").cast("int"))\
    .withColumn("unit_price", col("unit_price").cast("double"))

CONVERT DATES

In [0]:
from pyspark.sql.functions import to_date, col

df = df.withColumn("sale_date", to_date(col("sale_date"), "M/d/yyyy"))
display(df)


REMOVE DUPLICATES

In [0]:
df = df.dropDuplicates(["sale_id"])

In [0]:
df = df.withColumn("total_amount", col("quantity") * col("unit_price"))

Quality Rules

In [0]:
# A null quantity is rejected rather than disappearing from both groups.
from pyspark.sql.functions import coalesce, lit
valid_quantity = coalesce(col("quantity") > 0, lit(False))
df_valid = df.filter(valid_quantity)
df_invalid = df.filter(~valid_quantity)

In [0]:
print("Total Records", df.count())
print("Valid Records", df_valid.count())
print("Invalid Records", df_invalid.count())

In [0]:
storage_account = "strsalesdata"
container = "csalesdata"

# Container SAS with Create and Write permissions. Never paste it into source code.
sas_params = dbutils.secrets.get(scope="retail-sales", key="output-sas").strip().lstrip("?")

In [0]:
# Small-data demonstration: toPandas() collects all valid rows on the driver.
csv_data = df_valid.toPandas().to_csv(index=False).encode("utf-8")
upload_url = (
    f"https://{storage_account}.blob.core.windows.net/"
    f"{container}/valid/valid_sales.csv?{sas_params}"
)
response = requests.put(
    upload_url, data=csv_data,
    headers={"x-ms-blob-type": "BlockBlob", "Content-Type": "text/csv"},
    timeout=120
)
if response.status_code != 201:
    raise RuntimeError(f"Upload failed: HTTP {response.status_code}")
print("Valid records uploaded successfully")